# Rearranging Reports

In [ ]:
import hashlib
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import pandas as pd

# ---------- Settings ----------
ROOT = Path.cwd()
if not (ROOT / "reports").exists():      # in case the notebook runs from notebooks/
    ROOT = ROOT.parent

SCAN_DIRS = [
    ROOT / "reports" / "transformer",
    ROOT / "reports" / "transformer_stage2",
]

def sha256_of(path: Path, chunk=1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

# ---------- 1) Collect all files ----------
rows = []
for base in SCAN_DIRS:
    if not base.exists():
        print(f"WARNING: folder does not exist: {base}")
        continue
    for p in sorted(base.rglob("*")):
        if p.is_file():
            st = p.stat()
            rows.append({
                "root": base.name,                              # transformer or transformer_stage2
                "subfolder": str(p.parent.relative_to(base)) or ".",
                "file": p.name,
                "size_bytes": st.st_size,
                "size_MB": round(st.st_size / 1e6, 3),
                "modified": datetime.fromtimestamp(st.st_mtime).strftime("%Y-%m-%d %H:%M"),
                "path": p,
            })

df = pd.DataFrame(rows)
if df.empty:
    raise SystemExit("No files found; check the ROOT path.")

# ---------- 2) Hash only files that share a size with another file ----------
by_size = defaultdict(list)
for i, s in df["size_bytes"].items():
    by_size[s].append(i)

df["sha256"] = None
for s, idxs in by_size.items():
    if len(idxs) > 1:                                           # duplicate candidates
        for i in idxs:
            df.at[i, "sha256"] = sha256_of(df.at[i, "path"])

# ---------- 3) Group duplicates ----------
hash_counts = df["sha256"].value_counts()
dup_hashes = hash_counts[hash_counts > 1].index.tolist()
group_map = {h: f"D{n:02d}" for n, h in enumerate(dup_hashes, 1)}
df["dup_group"] = df["sha256"].map(group_map)
df["sha_short"] = df["sha256"].str[:10]

# ---------- 4) Print tables ----------
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 60)
cols = ["root", "subfolder", "file", "size_MB", "modified", "dup_group", "sha_short"]

print("=" * 100)
print("1) Summary per root folder")
print("=" * 100)
summary = df.groupby("root").agg(files=("file", "count"), total_MB=("size_MB", "sum")).round(2)
print(summary.to_string())

print("\n" + "=" * 100)
print("2) All files (empty dup_group = unique)")
print("=" * 100)
print(df.sort_values(["root", "subfolder", "file"])[cols].fillna("").to_string(index=False))

print("\n" + "=" * 100)
print("3) Duplicate groups (byte-identical content)")
print("=" * 100)
dups = df[df["dup_group"].notna()].sort_values(["dup_group", "root", "subfolder", "file"])
if dups.empty:
    print("No duplicate files found.")
else:
    print(dups[["dup_group"] + cols[:5] + ["sha_short"]].to_string(index=False))
    wasted = sum(
        g["size_bytes"].iloc[0] * (len(g) - 1)
        for _, g in dups.groupby("dup_group")
    )
    print(f"\n{dups['dup_group'].nunique()} duplicate groups | "
          f"{len(dups)} files involved | wasted space ~ {wasted/1e6:.2f} MB")

# ---------- 5) Same name, different content (important for the mix-up!) ----------
print("\n" + "=" * 100)
print("4) Same file name with different content (possible overwrite/collision)")
print("=" * 100)
same_name = df.groupby("file").filter(lambda g: len(g) > 1)
same_name = same_name[~same_name["sha256"].duplicated(keep=False) | same_name["sha256"].isna()]
if same_name.empty:
    print("None found.")
else:
    print(same_name.sort_values(["file", "root"])[cols].fillna("").to_string(index=False))